# Sentinel-2 年度合成与分块导出

读取本地研究区和瓦片矢量，生成指定年份的 Sentinel-2 地表反射率合成影像，并按瓦片导出到 Google Drive。

（1）初始化 Earth Engine，设置年份、文件路径、瓦片编号范围和导出目录。

（2）通过 geemap 读取本地研究区和瓦片文件，按编号筛选待导出的瓦片。

（3）筛选年度 Sentinel-2 影像，关联 Cloud Score+，保留清晰像元及六个目标波段。

（4）分别计算夏季（6—9 月）和其他月份的中值合成，以其他月份补齐夏季空缺，无数据像元填 0。

（5）生成像元来源图层（0：无数据；1：夏季；2：其他月份），通过 geemap 查看影像和瓦片边界。

（6）在瓦片循环中直接创建并启动 10 m GeoTIFF 导出任务，到 Earth Engine Tasks 查看进度。

In [ ]:
# 1. 初始化 Earth Engine

from pathlib import Path

import ee
import geemap

MY_PROJECT_ID = 'your-project-id'  # 替换为你的 Earth Engine 项目 ID。

try:
    ee.Initialize(project=MY_PROJECT_ID)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)

print(f"Earth Engine 初始化完成：{MY_PROJECT_ID}")

In [ ]:
# 2. 参数设置

BOUNDARY_LOCAL_PATH = Path('data/hma/hma.gpkg')
TILES_LOCAL_PATH = Path('data/hma_tiles/hma_tiles_buf.gpkg')

TARGET_YEAR = 2020
START_ID = 110  # 瓦片编号范围，两端均包含。
END_ID = 112
EXPORT_FOLDER = 'HMA_S2'  # Google Drive 导出目录。

BANDS_S2 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # 蓝、绿、红、近红外、两个短波红外波段。
MAX_CLOUD_PERCENT = 80  # 影像级云量筛选阈值。
CS_THRESHOLD = 0.65  # 像元级 Cloud Score+ 清晰度阈值。

In [ ]:
# 3. 读取本地矢量并筛选瓦片

for vector_path in (BOUNDARY_LOCAL_PATH, TILES_LOCAL_PATH):
    if not vector_path.is_file():
        raise FileNotFoundError(f"矢量文件不存在，请检查路径：{vector_path}")

if START_ID > END_ID:
    raise ValueError("START_ID 不能大于 END_ID。")

study_area = geemap.vector_to_ee(str(BOUNDARY_LOCAL_PATH))
tiles = geemap.vector_to_ee(str(TILES_LOCAL_PATH))

# 将字符串编号转为数值，避免字符串排序影响范围筛选。
selected_tiles = (
    tiles.map(lambda feature: feature.set(
        'tile_id_num', ee.Number.parse(feature.get('tile_id'))
    ))
    .filter(ee.Filter.gte('tile_id_num', START_ID))
    .filter(ee.Filter.lte('tile_id_num', END_ID))
    .sort('tile_id_num')
)

In [ ]:
# 4. 筛选年度影像并去云

s2_annual = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PERCENT))
    .filterBounds(study_area)
)

cs_plus_annual = (
    ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
    .filterBounds(study_area)
)

# 按 system:index 关联 Cloud Score+，保留 cs 达到阈值的像元。
s2_with_cs = s2_annual.linkCollection(cs_plus_annual, ['cs'])
s2_masked_annual = (
    s2_with_cs.map(
        lambda image: image.updateMask(image.select('cs').gte(CS_THRESHOLD))
    )
    .select(BANDS_S2)
)

In [ ]:
# 5. 季节合成与像元来源

# 夏季：6—9 月；其他月份：1—5 月和 10—12 月。
s2_summer = (
    s2_masked_annual
    .filter(ee.Filter.calendarRange(6, 9, 'month'))
    .median()
)

non_summer_filter = ee.Filter.Or(
    ee.Filter.calendarRange(1, 5, 'month'),
    ee.Filter.calendarRange(10, 12, 'month'),
)
s2_other_months = s2_masked_annual.filter(non_summer_filter).median()

# 仅在夏季无有效像元的位置使用其他月份，最终无数据填 0。
combined_img = s2_summer.unmask(s2_other_months)
final_img = combined_img.toUint16().unmask(0)

In [ ]:
source_summer = ee.Image.constant(1).updateMask(s2_summer.select('B2').mask())
source_other = ee.Image.constant(2).updateMask(s2_other_months.select('B2').mask())

# 0：无数据；1：夏季；2：其他月份。与影像采用相同的填补顺序。
source_map = source_summer.unmask(source_other).unmask(0).toByte()

In [ ]:
# 6. 可视化检查

m = geemap.Map()
m.centerObject(study_area, 5)

vis_true = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
vis_false = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}
source_vis = {
    'min': 0,
    'max': 2,
    'palette': ['000000', '00FF00', 'FFFF00'],
}

m.addLayer(s2_summer.clipToCollection(study_area), vis_false, '夏季假彩色', False)
# m.addLayer(final_img.clipToCollection(study_area), vis_true, '合成真彩色', False)
m.addLayer(final_img.clipToCollection(study_area), vis_false, '合成假彩色', True)
# m.addLayer(source_map.clipToCollection(study_area), source_vis, '像元来源', True)
m.addLayer(selected_tiles, {'color': 'FF0000'}, '待导出瓦片', True)

display(m)

In [ ]:
# # 7. 按瓦片导出到 Google Drive

# tile_features = selected_tiles.getInfo()['features']
# print(f"待导出瓦片数：{len(tile_features)}")

# for tile_feature in tile_features:
#     properties = tile_feature['properties']
#     tile_id = str(properties['tile_id'])
#     tile_crs = str(properties['proj']).replace(' ', '')
#     tile_geometry = ee.Geometry(tile_feature['geometry'])
#     export_name = f'tibet_s2_{TARGET_YEAR}_tile_{tile_id}'

#     task = ee.batch.Export.image.toDrive(
#         image=final_img,
#         description=export_name,
#         folder=EXPORT_FOLDER,
#         fileNamePrefix=export_name,
#         region=tile_geometry,
#         crs=tile_crs,
#         scale=10,
#         maxPixels=1e13,
#         fileFormat='GeoTIFF',
#     )
#     task.start()
#     print(f"已提交瓦片 {tile_id} 的导出任务，Task ID：{task.id}")

# print("导出任务提交结束，请到 Earth Engine Tasks 查看进度。")